# Tempo de UTI em insuficiência cardíaca: mais variáveis das 24 horas e coorte do artigo de Yang

**Duas mudanças, testadas separadas e juntas**

1. **Mais variáveis, todas das primeiras 24 horas** (`[intime, intime + 24 h)`): ventilação invasiva, PEEP, FiO2, escala de Glasgow, temperatura, lactato, bicarbonato, leucócitos, hemoglobina, plaquetas, INR, bilirrubina, NT-proBNP, troponina T, pH e pCO2.
2. **Coorte como no artigo de Yang et al. (2025):** só a primeira UTI de cada paciente e sem óbitos hospitalares.

**Quatro variantes, com o mesmo método**

| Variante | Coorte | Variáveis |
|---|---|---|
| A | todas as estadias de UTI com IC | as 24 de antes |
| B | todas as estadias de UTI com IC | as de antes + as novas |
| C | primeira UTI e sobreviventes | as 24 de antes |
| D | primeira UTI e sobreviventes | as de antes + as novas |

**Regras**

- O CID só define a coorte. Não entra como variável e não é o alvo.
- Nada depois de `intime + 24 h` entra em X.
- Divisão por paciente (`GroupShuffleSplit`, `random_state=42`, 80/20). Imputação e one-hot ajustados só no treino.
- Hiperparâmetros fixos, iguais aos escolhidos no treino da etapa anterior. O teste não participou de nenhuma escolha.
- Para cada variante: `XGBClassifier` (AUC, para estadia > 3,55 dias e > 7 dias) e `XGBRegressor` (MAE e R² em dias, treinado em `log(1 + dias)`).

**Atenção sobre a coorte de Yang:** excluir óbitos usa um desfecho que só se conhece no fim da internação. Serve para definir a população do estudo, não para uma ferramenta usada na admissão. Os resultados de A e C não são comparáveis diretamente, porque os pacientes de teste são outros.

**O que ficou de fora:** vasopressores, diuréticos e balanço hídrico ficam nas tabelas `inputevents` e `outputevents` do MIMIC-IV, que ainda não estão em `data/mimic-iv/`. A última seção explica como incluir.

**Tempo:** a primeira execução lê `chartevents` e `labevents` (cerca de 45 minutos). O resultado fica em `artefatos_24h_ampliado/tabela_24h_ampliada.csv`. Depois disso, o caderno roda em poucos minutos.

Pré-requisito: `artefatos/coorte_bruta.csv`, gerado pelo pré-processamento.

## 1. Configuração

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier, XGBRegressor

RANDOM_STATE = 42
RAIZ = Path.cwd()
MIMIC = RAIZ / "data" / "mimic-iv"
COORTE = RAIZ / "artefatos" / "coorte_bruta.csv"
SAIDA = RAIZ / "artefatos_24h_ampliado"
SAIDA.mkdir(exist_ok=True)
CACHE = SAIDA / "tabela_24h_ampliada.csv"
HORAS = 24
CORTES_DIAS = [3.55, 7.0]
N_BOOTSTRAP = 200

# chartevents (módulo icu do MIMIC-IV)
ITENS_CHART = {
    223761: "temp_f", 223762: "temp_c",
    220739: "gcs_olho", 223900: "gcs_verbal", 223901: "gcs_motor",
    223835: "fio2",
    220339: "peep", 224684: "vt_programado", 224685: "vt_observado",
}
ITENS_VENTILADOR = {220339, 224684, 224685}

# labevents (módulo hosp): itemid -> (nome, resumo na janela de 24 h, faixa plausível)
LABS = {
    50813: ("lactato", "max", (0.2, 30.0)),
    50882: ("bicarbonato", "min", (5.0, 50.0)),
    51301: ("leucocitos", "max", (0.1, 200.0)),
    51222: ("hemoglobina", "min", (3.0, 22.0)),
    51265: ("plaquetas", "min", (5.0, 1500.0)),
    51237: ("inr", "max", (0.5, 15.0)),
    50885: ("bilirrubina", "max", (0.1, 60.0)),
    50963: ("nt_probnp", "max", (5.0, 70000.0)),
    51003: ("troponina_t", "max", (0.001, 50.0)),
    50820: ("ph", "min", (6.8, 7.8)),
    50818: ("pco2", "max", (10.0, 200.0)),
}

BASE_NUM = [
    "idade", "pam_24h", "frequencia_cardiaca_24h", "frequencia_respiratoria_24h", "spo2_24h",
    "creatinina_24h", "ureia_24h", "sodio_24h", "potassio_24h", "glicose_24h", "ptt_24h", "feve",
]
BASE_BIN = ["feve_ausente"]
NOVAS_NUM = [
    "temperatura_24h", "gcs_min_24h", "fio2_max_24h", "peep_max_24h",
    "lactato_max_24h", "bicarbonato_min_24h", "leucocitos_max_24h", "hemoglobina_min_24h",
    "plaquetas_min_24h", "inr_max_24h", "bilirrubina_max_24h", "nt_probnp_max_24h",
    "troponina_t_max_24h", "ph_min_24h", "pco2_max_24h",
]
NOVAS_BIN = [
    "ventilacao_invasiva_24h", "lactato_medido", "nt_probnp_medido", "troponina_medida", "gasometria_medida",
]
CATEGORICAS = ["genero", "tipo_admissao"]

print("Coorte:", COORTE, "| existe:", COORTE.exists())
print("Cache:", CACHE, "| existe:", CACHE.exists())

## 2. Leitura das novas variáveis nas primeiras 24 horas

Cada linha lida só vale se `intime <= charttime < intime + 24 h`. Valores fora de faixa plausível viram ausentes.

In [ ]:
def achar(nome):
    for caminho in (MIMIC / nome, MIMIC / f"{nome}.gz", MIMIC / "icu" / nome, MIMIC / "hosp" / nome):
        if caminho.exists():
            return caminho
    raise FileNotFoundError(nome)


def varrer_chartevents(estadias):
    inicio = estadias.set_index("stay_id")["intime"]
    ids = set(inicio.index.astype(np.int64))
    partes, lidas = [], 0
    leitor = pd.read_csv(
        achar("chartevents.csv"), usecols=["stay_id", "charttime", "itemid", "valuenum"],
        chunksize=1_000_000, low_memory=False,
    )
    for bloco in leitor:
        lidas += len(bloco)
        item = pd.to_numeric(bloco["itemid"], errors="coerce")
        estadia = pd.to_numeric(bloco["stay_id"], errors="coerce")
        valor = pd.to_numeric(bloco["valuenum"], errors="coerce")
        ficar = item.isin(list(ITENS_CHART)) & estadia.isin(ids) & valor.notna()
        if ficar.any():
            r = pd.DataFrame({
                "stay_id": estadia[ficar].astype(np.int64).to_numpy(),
                "charttime": pd.to_datetime(bloco.loc[ficar, "charttime"], errors="coerce").to_numpy(),
                "itemid": item[ficar].astype(np.int64).to_numpy(),
                "valuenum": valor[ficar].to_numpy(),
            })
            r["intime"] = r["stay_id"].map(inicio)
            dentro = (r["charttime"] >= r["intime"]) & (r["charttime"] < r["intime"] + pd.Timedelta(hours=HORAS))
            partes.append(r.loc[dentro, ["stay_id", "itemid", "valuenum"]])
        if lidas % 50_000_000 == 0:
            print(f"  chartevents: {lidas:,} linhas lidas", flush=True)
    return pd.concat(partes, ignore_index=True)


def resumir_chartevents(bruto, ids_estadia):
    saida = pd.DataFrame(index=pd.Index(ids_estadia, name="stay_id"))
    bruto = bruto.assign(nome=bruto["itemid"].map(ITENS_CHART))

    temp = bruto[bruto["nome"].isin(["temp_f", "temp_c"])].copy()
    temp["celsius"] = np.where(temp["nome"] == "temp_f", (temp["valuenum"] - 32.0) / 1.8, temp["valuenum"])
    temp = temp[(temp["celsius"] >= 30.0) & (temp["celsius"] <= 43.0)]
    saida["temperatura_24h"] = temp.groupby("stay_id")["celsius"].median()

    def pior(nome, minimo, maximo):
        parte = bruto[bruto["nome"] == nome]
        parte = parte[(parte["valuenum"] >= minimo) & (parte["valuenum"] <= maximo)]
        return parte.groupby("stay_id")["valuenum"].min()

    saida["gcs_min_24h"] = pior("gcs_olho", 1, 4) + pior("gcs_verbal", 1, 5) + pior("gcs_motor", 1, 6)

    fio2 = bruto[bruto["nome"] == "fio2"].copy()
    fio2["valuenum"] = np.where(fio2["valuenum"] <= 1.0, fio2["valuenum"] * 100.0, fio2["valuenum"])
    fio2 = fio2[(fio2["valuenum"] >= 21.0) & (fio2["valuenum"] <= 100.0)]
    saida["fio2_max_24h"] = fio2.groupby("stay_id")["valuenum"].max()

    peep = bruto[(bruto["nome"] == "peep") & (bruto["valuenum"] >= 0) & (bruto["valuenum"] <= 30)]
    saida["peep_max_24h"] = peep.groupby("stay_id")["valuenum"].max()

    ventilados = set(bruto.loc[bruto["itemid"].isin(ITENS_VENTILADOR), "stay_id"])
    saida["ventilacao_invasiva_24h"] = saida.index.isin(ventilados).astype(int)
    return saida


def varrer_labevents(estadias):
    janela = estadias[["subject_id", "stay_id", "intime"]].copy()
    janela["subject_id"] = janela["subject_id"].astype(np.int64)
    janela["fim"] = janela["intime"] + pd.Timedelta(hours=HORAS)
    ids = set(janela["subject_id"])
    minimos = {k: v[2][0] for k, v in LABS.items()}
    maximos = {k: v[2][1] for k, v in LABS.items()}
    partes, lidas = [], 0
    leitor = pd.read_csv(
        achar("labevents.csv"), usecols=["subject_id", "charttime", "itemid", "valuenum"],
        chunksize=1_000_000, low_memory=False,
    )
    for bloco in leitor:
        lidas += len(bloco)
        item = pd.to_numeric(bloco["itemid"], errors="coerce")
        paciente = pd.to_numeric(bloco["subject_id"], errors="coerce")
        valor = pd.to_numeric(bloco["valuenum"], errors="coerce")
        ficar = item.isin(list(LABS)) & paciente.isin(ids) & valor.notna()
        if ficar.any():
            r = pd.DataFrame({
                "subject_id": paciente[ficar].astype(np.int64).to_numpy(),
                "charttime": pd.to_datetime(bloco.loc[ficar, "charttime"], errors="coerce").to_numpy(),
                "itemid": item[ficar].astype(np.int64).to_numpy(),
                "valuenum": valor[ficar].to_numpy(),
            })
            r = r.merge(janela, on="subject_id", how="inner")
            r = r[(r["charttime"] >= r["intime"]) & (r["charttime"] < r["fim"])]
            r = r[(r["valuenum"] >= r["itemid"].map(minimos)) & (r["valuenum"] <= r["itemid"].map(maximos))]
            partes.append(r[["stay_id", "itemid", "valuenum"]])
        if lidas % 50_000_000 == 0:
            print(f"  labevents: {lidas:,} linhas lidas", flush=True)
    return pd.concat(partes, ignore_index=True)


def resumir_labevents(bruto, ids_estadia):
    saida = pd.DataFrame(index=pd.Index(ids_estadia, name="stay_id"))
    for item, (nome, resumo, _) in LABS.items():
        parte = bruto[bruto["itemid"] == item].groupby("stay_id")["valuenum"].agg(resumo)
        saida[f"{nome}_{resumo}_24h"] = parte
    return saida


def montar_tabela():
    coorte = pd.read_csv(COORTE)
    estadias = pd.read_csv(achar("icustays.csv"), usecols=["stay_id", "intime"])
    estadias["intime"] = pd.to_datetime(estadias["intime"])
    obitos = pd.read_csv(achar("admissions.csv"), usecols=["hadm_id", "hospital_expire_flag"])
    tabela = coorte.merge(estadias, on="stay_id", how="inner").merge(obitos, on="hadm_id", how="left")
    tabela = tabela.drop_duplicates("stay_id").reset_index(drop=True)
    ids_estadia = tabela["stay_id"].astype(np.int64).to_numpy()
    janela = tabela[["subject_id", "stay_id", "intime"]]

    print("Lendo chartevents (primeiras 24 h)...", flush=True)
    chart = resumir_chartevents(varrer_chartevents(janela), ids_estadia)
    print("Lendo labevents (primeiras 24 h)...", flush=True)
    labs = resumir_labevents(varrer_labevents(janela), ids_estadia)
    novas = chart.join(labs)
    return tabela.merge(novas, left_on="stay_id", right_index=True, how="left")

In [ ]:
if CACHE.exists():
    tabela = pd.read_csv(CACHE, parse_dates=["intime"])
    print("Tabela lida do cache.")
else:
    tabela = montar_tabela()
    tabela.to_csv(CACHE, index=False, encoding="utf-8-sig")
    print("Tabela calculada e salva no cache.")

tabela["lactato_medido"] = tabela["lactato_max_24h"].notna().astype(int)
tabela["nt_probnp_medido"] = tabela["nt_probnp_max_24h"].notna().astype(int)
tabela["troponina_medida"] = tabela["troponina_t_max_24h"].notna().astype(int)
tabela["gasometria_medida"] = tabela["ph_min_24h"].notna().astype(int)
tabela["ventilacao_invasiva_24h"] = tabela["ventilacao_invasiva_24h"].fillna(0).astype(int)
tabela = tabela.dropna(subset=["los_dias", "subject_id"])
tabela = tabela[tabela["los_dias"] > 0].copy()
print(f"Estadias: {len(tabela):,} | pacientes: {tabela['subject_id'].nunique():,}")

### Conferência das novas variáveis

Confira se a cobertura e os valores típicos fazem sentido clínico (por exemplo, lactato mediano perto de 1,5 a 2,5 mmol/L, temperatura perto de 36,5 a 37 °C, Glasgow perto de 12 a 15). Se algum número vier absurdo, o `itemid` pode estar errado.

In [ ]:
conferencia = []
for coluna in NOVAS_NUM:
    serie = tabela[coluna]
    conferencia.append({
        "variavel": coluna,
        "pct_presente": round(100 * serie.notna().mean(), 1),
        "p5": round(serie.quantile(0.05), 2),
        "mediana": round(serie.median(), 2),
        "p95": round(serie.quantile(0.95), 2),
    })
conferencia = pd.DataFrame(conferencia)
conferencia.to_csv(SAIDA / "conferencia_novas_variaveis.csv", index=False, encoding="utf-8-sig")
display(conferencia)
print(f"Ventilação invasiva nas 24 h: {100 * tabela['ventilacao_invasiva_24h'].mean():.1f}% das estadias")

## 3. As duas coortes

- **Todas:** todas as estadias de UTI com IC (internações com I50 ou 428 no CID, só para filtrar).
- **Yang:** a primeira UTI de cada paciente (pela data de entrada, dentro da coorte de IC) e, entre elas, só quem não morreu no hospital (`hospital_expire_flag = 0`).

In [ ]:
todas = tabela.copy()
primeira = tabela.sort_values("intime").drop_duplicates("subject_id", keep="first")
yang = primeira[primeira["hospital_expire_flag"] == 0].copy()

linhas = []
for nome, base in [("Todas as estadias", todas), ("Primeira UTI, sem óbitos (Yang)", yang)]:
    linha = {
        "coorte": nome,
        "estadias": len(base),
        "pacientes": base["subject_id"].nunique(),
        "los_mediano_dias": round(base["los_dias"].median(), 2),
        "los_p75_dias": round(base["los_dias"].quantile(0.75), 2),
    }
    for corte in CORTES_DIAS:
        linha[f"pct_mais_de_{corte}_dias"] = round(100 * (base["los_dias"] > corte).mean(), 1)
    linhas.append(linha)
display(pd.DataFrame(linhas))
print(f"Mortes hospitalares na coorte inteira: {int((tabela['hospital_expire_flag'] == 1).sum()):,}")

## 4. Preparação e modelos

A mesma função serve às quatro variantes. Mediana de imputação e categorias vêm só do treino.

In [ ]:
def preparar(base, colunas_num, colunas_bin):
    divisor = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
    i_treino, i_teste = next(divisor.split(base, groups=base["subject_id"]))
    treino, teste = base.iloc[i_treino].copy(), base.iloc[i_teste].copy()
    assert not set(treino["subject_id"]) & set(teste["subject_id"]), "paciente repetido"

    for c in CATEGORICAS:
        treino[c] = treino[c].astype("string").fillna("desconhecido")
        teste[c] = teste[c].astype("string").fillna("desconhecido")
    try:
        codificador = OneHotEncoder(sparse_output=False, handle_unknown="ignore", dtype=np.float64)
    except TypeError:
        codificador = OneHotEncoder(sparse=False, handle_unknown="ignore", dtype=np.float64)
    d_treino = codificador.fit_transform(treino[CATEGORICAS])
    nomes = codificador.get_feature_names_out(CATEGORICAS)
    d_treino = pd.DataFrame(d_treino, columns=nomes, index=treino.index)
    d_teste = pd.DataFrame(codificador.transform(teste[CATEGORICAS]), columns=nomes, index=teste.index)

    imputador = SimpleImputer(strategy="median")
    n_treino = pd.DataFrame(imputador.fit_transform(treino[colunas_num]), columns=colunas_num, index=treino.index)
    n_teste = pd.DataFrame(imputador.transform(teste[colunas_num]), columns=colunas_num, index=teste.index)

    x_treino = pd.concat([n_treino, treino[colunas_bin].astype(float), d_treino], axis=1)
    x_teste = pd.concat([n_teste, teste[colunas_bin].astype(float), d_teste], axis=1)
    x_teste = x_teste.reindex(columns=x_treino.columns, fill_value=0.0)
    proibidas = [c for c in x_treino.columns if "los" in c or "icd" in c.lower() or "diag" in c.lower()]
    assert not proibidas, f"coluna proibida em X: {proibidas}"
    return treino, teste, x_treino, x_teste


def intervalo_auc(y, prob, grupos, repeticoes=N_BOOTSTRAP):
    grupos = np.asarray(grupos)
    ordem = np.argsort(grupos, kind="stable")
    _, inicio = np.unique(grupos[ordem], return_index=True)
    por_paciente = np.split(ordem, inicio[1:])
    sorteio = np.random.default_rng(RANDOM_STATE)
    aucs = []
    for _ in range(repeticoes):
        escolha = sorteio.integers(0, len(por_paciente), len(por_paciente))
        indices = np.concatenate([por_paciente[i] for i in escolha])
        if y[indices].min() == y[indices].max():
            continue
        aucs.append(roc_auc_score(y[indices], prob[indices]))
    return tuple(np.percentile(aucs, [2.5, 97.5]))


def avaliar_variante(rotulo, base, colunas_num, colunas_bin):
    treino, teste, x_treino, x_teste = preparar(base, colunas_num, colunas_bin)
    linhas, guardado = [], {}
    for corte in CORTES_DIAS:
        y_treino = (treino["los_dias"] > corte).astype(int).to_numpy()
        y_teste = (teste["los_dias"] > corte).astype(int).to_numpy()
        modelo = XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
            min_child_weight=5, scale_pos_weight=float((y_treino == 0).sum() / (y_treino == 1).sum()),
            objective="binary:logistic", eval_metric="auc", n_jobs=1, random_state=RANDOM_STATE,
        )
        modelo.fit(x_treino, y_treino)
        prob = modelo.predict_proba(x_teste)[:, 1]
        baixo, alto = intervalo_auc(y_teste, prob, teste["subject_id"])
        linhas.append({
            "variante": rotulo, "tarefa": f"mais de {corte} dias",
            "AUC": roc_auc_score(y_teste, prob), "AUC_IC95_baixo": baixo, "AUC_IC95_alto": alto,
            "AUC_PR": average_precision_score(y_teste, prob), "prevalencia": y_teste.mean(),
            "n_teste": len(teste),
        })
        if corte == CORTES_DIAS[0]:
            guardado = {"modelo": modelo, "x_teste": x_teste}

    regressor = XGBRegressor(
        n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, objective="reg:squarederror", n_jobs=1, random_state=RANDOM_STATE,
    )
    regressor.fit(x_treino, np.log1p(treino["los_dias"].to_numpy()))
    previsto = np.clip(np.expm1(regressor.predict(x_teste)), 0, None)
    linhas.append({
        "variante": rotulo, "tarefa": "regressão (dias)",
        "MAE": mean_absolute_error(teste["los_dias"], previsto),
        "R2": r2_score(teste["los_dias"], previsto),
        "n_teste": len(teste),
    })
    return linhas, guardado

In [ ]:
numericas_base = BASE_NUM
numericas_todas = BASE_NUM + NOVAS_NUM
binarias_base = BASE_BIN
binarias_todas = BASE_BIN + NOVAS_BIN

linhas_total, guardados = [], {}
for rotulo, base, num, bin_ in [
    ("A: todas, variáveis de antes", todas, numericas_base, binarias_base),
    ("B: todas, com novas variáveis", todas, numericas_todas, binarias_todas),
    ("C: Yang, variáveis de antes", yang, numericas_base, binarias_base),
    ("D: Yang, com novas variáveis", yang, numericas_todas, binarias_todas),
]:
    print(f"Treinando {rotulo}...", flush=True)
    linhas, guardado = avaliar_variante(rotulo, base, num, bin_)
    linhas_total.extend(linhas)
    guardados[rotulo] = guardado

resultados = pd.DataFrame(linhas_total)
resultados.to_csv(SAIDA / "resultados_variantes.csv", index=False, encoding="utf-8-sig")

## 5. Resultados

In [ ]:
classificacao = resultados[resultados["tarefa"].str.startswith("mais de")].copy()
classificacao["AUC (IC 95%)"] = classificacao.apply(
    lambda l: f"{l['AUC']:.3f} ({l['AUC_IC95_baixo']:.3f} a {l['AUC_IC95_alto']:.3f})", axis=1
)
classificacao["prevalência"] = (100 * classificacao["prevalencia"]).round(1).astype(str) + "%"
display(classificacao[["variante", "tarefa", "n_teste", "prevalência", "AUC (IC 95%)", "AUC_PR"]].round(3).reset_index(drop=True))

regressao = resultados[resultados["tarefa"] == "regressão (dias)"][["variante", "n_teste", "MAE", "R2"]]
display(regressao.round(3).reset_index(drop=True))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for eixo, corte in zip(eixos, CORTES_DIAS):
    parte = classificacao[classificacao["tarefa"] == f"mais de {corte} dias"].reset_index(drop=True)
    y = np.arange(len(parte))[::-1]
    erro = [parte["AUC"] - parte["AUC_IC95_baixo"], parte["AUC_IC95_alto"] - parte["AUC"]]
    eixo.errorbar(parte["AUC"], y, xerr=erro, fmt="o", color="#1f4e79", capsize=4)
    eixo.axvline(0.5, color="gray", linestyle="--", linewidth=1)
    eixo.set_yticks(y)
    eixo.set_yticklabels(parte["variante"])
    eixo.set_title(f"Estadia > {corte} dias")
    eixo.set_xlabel("AUC no teste, com IC 95%")
    eixo.set_xlim(0.45, 1.0)
plt.tight_layout()
plt.savefig(SAIDA / "auc_variantes.png", dpi=150)
plt.show()

## 6. Que variáveis pesam mais (SHAP, variante D, estadia > 3,55 dias)

As variáveis novas aparecem com a marca ★.

In [ ]:
rotulo_d = "D: Yang, com novas variáveis"
modelo_d, x_teste_d = guardados[rotulo_d]["modelo"], guardados[rotulo_d]["x_teste"]
amostra = x_teste_d.sample(n=min(2000, len(x_teste_d)), random_state=RANDOM_STATE)
valores = np.asarray(shap.TreeExplainer(modelo_d)(amostra).values)
pesos = (
    pd.DataFrame({"variavel": amostra.columns, "shap_medio_abs": np.abs(valores).mean(axis=0)})
    .sort_values("shap_medio_abs", ascending=False)
    .reset_index(drop=True)
)
pesos["nova"] = pesos["variavel"].isin(NOVAS_NUM + NOVAS_BIN)
pesos.to_csv(SAIDA / "shap_variante_D.csv", index=False, encoding="utf-8-sig")

topo = pesos.head(15).iloc[::-1]
rotulos = [("★ " if nova else "") + v for v, nova in zip(topo["variavel"], topo["nova"])]
plt.figure(figsize=(8, 6))
plt.barh(rotulos, topo["shap_medio_abs"], color=["#c0504d" if n else "#1f4e79" for n in topo["nova"]])
plt.xlabel("SHAP médio absoluto")
plt.title("Variante D, estadia > 3,55 dias")
plt.tight_layout()
plt.savefig(SAIDA / "shap_variante_D.png", dpi=150)
plt.show()
display(pesos.head(10))

## 7. O que ainda falta: vasopressores, diuréticos e balanço hídrico

Essas informações estão em tabelas do módulo `icu` do MIMIC-IV que ainda não estão na pasta `data/mimic-iv/`:

- `inputevents.csv.gz`: vasopressores (noradrenalina, adrenalina, vasopressina, dopamina, dobutamina, fenilefrina), furosemida e volume infundido.
- `outputevents.csv.gz`: diurese.
- `d_items.csv.gz`: dicionário dos `itemid`, para conferir os códigos.

O download é pelo navegador, na página do MIMIC-IV no PhysioNet, com o seu login. Com os arquivos na pasta, as variáveis entram na mesma janela de 24 horas.

## 8. Como ler

- Compare **A com B** e **C com D** para ver o efeito das variáveis novas. Compare **A com C** e **B com D** para ver o efeito da coorte, sabendo que os pacientes de teste são outros.
- O **intervalo de confiança** mostra a incerteza do teste. Se dois intervalos se sobrepõem bastante, a diferença pode ser acaso.
- **AUC-PR** deve ser lida contra a prevalência (o que um chute teria).
- A coorte de Yang exclui quem morreu. Isso é uma escolha de estudo e precisa aparecer assim no TCC.
- A AUC de 0,861 do artigo é de outra coorte (MIMIC-IV 2.2, outras variáveis). Serve como referência de ordem de grandeza.